In [1]:
from sklearn.datasets import fetch_20newsgroups
from sklearn.model_selection import train_test_split

news_data = fetch_20newsgroups(subset='all', remove=tuple(['headers', 'footers', 'quotes']), random_state=42)
print(type(news_data))
text_train, text_test, label_train, label_test = train_test_split(news_data.data, news_data.target, test_size=0.2, random_state=42)

<class 'sklearn.utils._bunch.Bunch'>


In [2]:
import re

def clean_text(text):
    # 1. 소문자화
    text = text.lower()
    # 2. 특수문자 제거 (단어가 아닌 문자 제거)
    text = re.sub(r'[^a-z\s]', '', text)
    # 3. 여러 공백을 하나로
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# 전체 데이터에 적용
text_train_cleaned = [clean_text(t) for t in text_train]
text_test_cleaned = [clean_text(t) for t in text_test]

In [3]:
text_train_cleaned[0]

'ive gotten very few posts on this group in the last couple days i recently added it to my feed list is it just me or is this group near death seen from the mailing list side im getting about the right amount of traffic patrick l mahan tgv window washer mahantgvcom waking a person unnecessarily should not be considered lazarus long a capital crime for a first offense that is from the notebooks of lazarus long patrick l mahan tgv window washer mahantgvcom'

In [29]:
import pandas as pd

text_train_df = pd.DataFrame({'text': text_train_cleaned, 'label': label_train})
text_test_df = pd.DataFrame({'text': text_test_cleaned, 'label': label_test})
text_train_df.head()

,text,label
0,ive gotten very few posts on this group in the...,5
1,interesting id fight the ticket first off ther...,8
2,i remember as a kid visiting my relatives on k...,13
3,it can be painless so it isnt cruel and it has...,0
4,the owners are whining about baseball not bein...,9


In [23]:
# 추론을 위해서 데이터 저장

text_train_df.to_csv("text_train.csv", index=False)
text_test_df.to_csv("text_test.csv", index=False)

In [30]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased", use_fast=True)

class NewsDataset(Dataset):
    def __init__(self, data, tokenizer, max_length=128):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        text = self.data.iloc[idx]['text']
        label = self.data.iloc[idx]['label']

        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        input_ids = encoding['input_ids'].squeeze(0)
        attention_mask = encoding['attention_mask'].squeeze(0)
        label = torch.tensor(label, dtype=torch.long)
        return input_ids, attention_mask, label

train_dataset = NewsDataset(text_train_df, tokenizer)
test_dataset = NewsDataset(text_test_df, tokenizer)

In [31]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [32]:
import torch
import torch.nn as nn
from transformers import DistilBertModel

class SimpleClassifier(nn.Module):
    def __init__(self, num_classes=20):
        super().__init__()
        self.bert = DistilBertModel.from_pretrained("distilbert-base-uncased")
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(self.bert.config.hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.last_hidden_state[:, 0]  # [CLS] 토큰에 해당하는 위치
        pooled_output = self.dropout(pooled_output)
        return self.fc(pooled_output)

In [37]:
from tqdm import tqdm

def train_model(model, train_loader, test_loader, optimizer, criterion, device, num_epochs=10, patience=3):
    model.to(device)

    best_loss = float('inf')
    patience_counter = 0

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0
        correct = 0
        total = 0

        for input_ids, attention_mask, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}", leave=False):
            input_ids = input_ids.to(device)
            attention_mask = attention_mask.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()
            outputs = model(input_ids, attention_mask)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * input_ids.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

        avg_train_loss = total_loss / total
        train_acc = correct / total
        print(f"[Epoch {epoch+1}] Train Loss: {avg_train_loss:.4f}, Accuracy: {train_acc:.4f}")

        val_loss, val_acc = evaluate_model(model, test_loader, criterion, device)
        print(f"[Epoch {epoch+1}] Val Loss: {val_loss:.4f}, Accuracy: {val_acc:.4f}")

        # EarlyStopping check
        if val_loss < best_loss:
            best_loss = val_loss
            patience_counter = 0
            torch.save(model.state_dict(), "nlp_weight.pt")
            print("→ Validation loss improved. Saving model.")
        else:
            patience_counter += 1
            print(f"→ No improvement. EarlyStopping patience: {patience_counter}/{patience}")
            if patience_counter >= patience:
                print("→ Early stopping triggered.")
                model.load_state_dict(torch.load("nlp_weight.pt", map_location=device))
                break

def evaluate_model(model, data_loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for input_ids, attention_mask, labels in tqdm(data_loader, leave=False):
            input_ids = input_ids.to(device)
            attention_mask = attention_mask.to(device)
            labels = labels.to(device)

            outputs = model(input_ids, attention_mask)
            loss = criterion(outputs, labels)

            total_loss += loss.item() * input_ids.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    avg_loss = total_loss / total
    acc = correct / total
    # print(f"[Validation/Test] Loss: {avg_loss:.4f}, Accuracy: {acc:.4f}")
    return avg_loss, acc

In [38]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = SimpleClassifier(num_classes=20)
model = model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
criterion = nn.CrossEntropyLoss()

In [39]:
train_model(model, train_loader, test_loader, optimizer, criterion, device, num_epochs=10, patience=3)

[Epoch 1] Train Loss: 1.4455, Accuracy: 0.5647


[Epoch 1] Val Loss: 1.0418, Accuracy: 0.6695
→ Validation loss improved. Saving model.


[Epoch 2] Train Loss: 0.8551, Accuracy: 0.7373


[Epoch 2] Val Loss: 0.9725, Accuracy: 0.6939
→ Validation loss improved. Saving model.


[Epoch 3] Train Loss: 0.6248, Accuracy: 0.8109


...

[Epoch 4] Val Loss: 1.0135, Accuracy: 0.7114
→ No improvement. EarlyStopping patience: 1/3


[Epoch 5] Train Loss: 0.3146, Accuracy: 0.9099


[Epoch 5] Val Loss: 1.0157, Accuracy: 0.7255
→ No improvement. EarlyStopping patience: 2/3


[Epoch 6] Train Loss: 0.2244, Accuracy: 0.9384


[Epoch 6] Val Loss: 1.0838, Accuracy: 0.7202
→ No improvement. EarlyStopping patience: 3/3
→ Early stopping triggered.


In [45]:
import torch.nn.utils.prune as prune

pruned_model = SimpleClassifier()
pruned_model.load_state_dict(torch.load("nlp_weight.pt", map_location="cpu"))

# classifier 내부 선형 계층 프루닝
modules_to_prune = [
    (pruned_model.fc, 'weight'),
]

# 전체 weight의 25%를 L1 기준으로 제거
prune.global_unstructured(
    modules_to_prune,
    pruning_method=prune.L1Unstructured,
    amount=0.25
)

# 마스크 적용 후, pruning 제거 (순수 weight만 남김)
for module, name in modules_to_prune:
    prune.remove(module, name)

# 저장
torch.save(pruned_model.state_dict(), "nlp_pruned.pt")

In [ ]:
# BERT 본체는 양자화 안 함, classifier만 적용
quantized_model = torch.quantization.quantize_dynamic(
    model, {nn.Linear}, dtype=torch.qint8
)

# 저장
torch.save(quantized_model, "nlp_quantized.pt")

In [49]:
!pip install onnx onnxruntime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 88.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.4/16.4 MB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 8.5 MB/s eta 0:00:00


In [50]:
import torch.onnx

# 1. 모델을 CPU로 옮김
model = model.to("cpu")
model.eval()

# 2. 입력도 CPU에 생성
dummy_input_ids = torch.randint(0, tokenizer.vocab_size, (1, 128)).to("cpu")
dummy_attention_mask = torch.ones(1, 128, dtype=torch.long).to("cpu")

# 3. ONNX Export
torch.onnx.export(
    model,
    (dummy_input_ids, dummy_attention_mask),
    "nlp_model.onnx",
    input_names=["input_ids", "attention_mask"],
    output_names=["logits"],
    dynamic_axes={
        "input_ids": {0: "batch_size", 1: "sequence_length"},
        "attention_mask": {0: "batch_size", 1: "sequence_length"},
        "logits": {0: "batch_size"}
    },
    opset_version=14,
    do_constant_folding=True
)